In [14]:
from pathlib import Path
Path.cwd()

WindowsPath('c:/Users/schneids/code/KlimSta-Brettspiel')

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from model.simulate import run_simulation

from model.strategies.random import choose_card as random_strategy
from model.strategies.electrician import choose_card as electrician_strategy
from model.strategies.off_gas import choose_card as decarbonizer_strategy
from model.strategies.insulator import choose_card as insulator_strategy
from model.strategies.people_pleaser import choose_card as pleaser_strategy


VERSION = Path("Versionen/paper_draft_v1")

EXPERIMENT_DIR = (
    VERSION
    / "results"
    / "strategy_comparison"
)

EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

N_GAMES = 10_000
SEED = 42

RULE = "random_draw"
DRAW_N = 20
PASS_PROBABILITY = 0.05
LOG_CHOICES = True

FORCE_RUN = True


STRATEGIES = [
    ("random", random_strategy),
    ("electrician", electrician_strategy),
    ("decarbonizer", decarbonizer_strategy),
    ("insulator", insulator_strategy),
    ("people pleaser", pleaser_strategy),
]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import json


games_results = []
plays_results = []

for strategy_name, strategy in STRATEGIES:

    print(f"\n--- {strategy_name} ---")

    strategy_dir = EXPERIMENT_DIR / strategy_name
    strategy_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    games_path = strategy_dir / "games.parquet"
    plays_path = strategy_dir / "plays.parquet"
    config_path = strategy_dir / "config.json"

    # Reuse complete existing run
    if not FORCE_RUN:
        if games_path.exists():

            games = pd.read_parquet(games_path)

            if len(games) == N_GAMES:
                print("Using existing results.")

                plays = (
                    pd.read_parquet(plays_path)
                    if plays_path.exists()
                    else pd.DataFrame()
                )

                games_results.append(games)

                if not plays.empty:
                    plays_results.append(plays)

                continue

    # Run simulation
    games, plays = run_simulation(
        VERSION,
        n_games=N_GAMES,
        strategy=strategy,
        strategy_name=strategy_name,
        rule=RULE,
        draw_n=DRAW_N,
        pass_probability=PASS_PROBABILITY,
        seed=SEED,
        save=False,
        log_choices=LOG_CHOICES,
    )

    # Experiment metadata
    games["experiment"] = "strategy_comparison"

    if not plays.empty:
        plays["experiment"] = "strategy_comparison"

    # Save this strategy immediately
    games.to_parquet(
        games_path,
        index=False,
    )

    if not plays.empty:
        plays.to_parquet(
            plays_path,
            index=False,
        )

    config = {
        "strategy": strategy_name,
        "n_games": N_GAMES,
        "seed": SEED,
        "rule": RULE,
        "draw_n": DRAW_N,
        "pass_probability": PASS_PROBABILITY,
    }

    config_path.write_text(
        json.dumps(
            config,
            indent=2,
        )
    )

    games_results.append(games)

    if not plays.empty:
        plays_results.append(plays)


--- random ---
Using existing results.

--- electrician ---
Using existing results.

--- decarbonizer ---
Using existing results.

--- insulator ---
Using existing results.

--- people pleaser ---
Using existing results.


In [17]:
all_games = pd.concat(
    games_results,
    ignore_index=True,
)

all_plays = pd.concat(
    plays_results,
    ignore_index=True,
)

all_games.to_parquet(
    EXPERIMENT_DIR / "all_games.parquet",
    index=False,
)

all_plays.to_parquet(
    EXPERIMENT_DIR / "all_plays.parquet",
    index=False,
)

print()
print(f"Saved {len(all_games):,} games")
print(f"Saved {len(all_plays):,} play records")


Saved 50,000 games
Saved 690,982 play records
